# Simulation Based Inference to Remove Sampling Bias - Real Data


Household infection model with parameters `alpha`, `beta`, `delta`, `mu_inf_SI`, `mu_inf_SC`, `mu_inf_AI`, `mu_inf_AC`, `mu_inf_AA`, `mu_susc_I`, `mu_susc_C`.

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'jax'

job_array_id = int(os.environ.get('SLURM_ARRAY_TASK_ID', 0))
n_procs = int(os.environ.get('SLURM_CPUS_PER_TASK', 1))
batch_size = 64

## Define model and prior

We fix some parameters such that the model becomes identifiable (checked with STAN). We fix the parameters `alpha`, `mu_protect_acq`, `mu_protect_transm`.

In [ ]:
import pickle
import itertools
from matplotlib import pyplot as plt

import numpy as np
import pandas as pd

import keras
import bayesflow as bf

from PedCov.stan import get_stan_posterior
from PedCov.simulator import OutbreakSimulator
from PedCov.helper_functions import plot_delay_distribution, plot_incubation_distribution, plot_generation_time_distribution, normalize_household_data, sampling_parameter_cis

In [ ]:
param_names = {  # comment out parameters that should not be estimated
    #'alpha': r'$\alpha$',
    'beta': r'$\beta$', 'delta': r'$\delta$',
    'mu_inf_SI': r'$\mu_\text{infectiousness}^\text{symptomatic Infant}$', 'mu_inf_SC': r'$\mu_\text{infectiousness}^\text{symptomatic Child}$',
    'mu_inf_AI': r'$\mu_\text{infectiousness}^\text{asymptomatic Infant}$', 'mu_inf_AC': r'$\mu_\text{infectiousness}^\text{asymptomatic Child}$', 'mu_inf_AA': r'$\mu_\text{infectiousness}^\text{asymptomatic Adult}$',
    'mu_susc_I': r'$\mu_\text{susceptibility}^\text{Infant}$', 'mu_susc_C': r'$\mu_\text{susceptibility}^\text{Child}$',
    #'mu_protect_acq': r'$\mu_\text{protect}^\text{acq}$', 'mu_protect_transm': r'$\mu_\text{protect}^\text{transm}$'
}

full_name_list = ['alpha', 'beta', 'delta',
                  'mu_inf_SI', 'mu_inf_SC', 'mu_inf_AI', 'mu_inf_AC', 'mu_inf_AA',
                  'mu_susc_I', 'mu_susc_C',
                  'mu_protect_acq', 'mu_protect_transm']

In [ ]:
# define the prior
def meta() -> dict:
    selection_procedure_id = np.random.choice([0, 1, 2])  # 0 for random, 1 for pedcov, 2 for adult
    variant_id = 0  # todo: np.random.choice([0, 1])  # 0 for alpha, 1 for omicron
    return dict(
        variant=['alpha', 'omicron'][variant_id],  # alpha or omicron
        variant_id=variant_id,
        selection_procedure=['random', 'pedcov', 'adultcov'][selection_procedure_id],
        selection_procedure_id=selection_procedure_id
    )

def prior(variant) -> dict:
    var = 0.7 # was 1 before
    if variant == 'alpha':
        alpha_fixed = 0.001  # fixed for alpha
    elif variant == 'omicron':
        alpha_fixed = 0.01  # fixed for omicron
    else:
        raise ValueError(f"Unknown variant: {variant}")

    params = {
        'alpha': np.random.uniform(0, 0.1) if 'alpha' in param_names.keys() else alpha_fixed,
        #'beta': np.random.uniform(0, 3) if 'beta' in param_names.keys() else 0.3,
        #'delta': np.random.uniform(-3, 3) if 'delta' in param_names.keys() else 0.1,
        #'alpha': np.random.gamma(shape=1.0, scale=1/20.0) if 'alpha' in param_names.keys() else 0.001,
        'beta': np.random.gamma(shape=2.0, scale=1/2.0) if 'beta' in param_names.keys() else 0.3,
        'delta': np.random.normal(0.0, 1.0) if 'delta' in param_names.keys() else 0.1,
        'mu_inf_SI': np.random.lognormal(0, var) if 'mu_inf_SI' in param_names.keys() else 1.0,
        'mu_inf_SC': np.random.lognormal(0, var) if 'mu_inf_SC' in param_names.keys() else 1.0,
        'mu_inf_AI': np.random.lognormal(0, var) if 'mu_inf_AI' in param_names.keys() else 1.0,
        'mu_inf_AC': np.random.lognormal(0, var) if 'mu_inf_AC' in param_names.keys() else 1.0,
        'mu_inf_AA': np.random.lognormal(0, var) if 'mu_inf_AA' in param_names.keys() else 1.0,
        'mu_susc_I': np.random.lognormal(0, var) if 'mu_susc_I' in param_names.keys() else 1.0,
        'mu_susc_C': np.random.lognormal(0, var) if 'mu_susc_C' in param_names.keys() else 1.0,
        'mu_protect_acq': np.random.lognormal(0, var) if 'mu_protect_acq' in param_names.keys() else 0.8,
        'mu_protect_transm': np.random.lognormal(0, var) if 'mu_protect_transm' in param_names.keys() else 1.0
    }
    return params


def plot_priors(n_samples=10000):
    # Sample from the prior
    samples = {key: np.zeros(n_samples) for key in param_names}
    for i in range(n_samples):
        p = prior("alpha")
        for key in samples:
            samples[key][i] = p[key]

    # Set up subplots
    fig, axes = plt.subplots(2, 5, figsize=(10, 4), tight_layout=True)
    axes = axes.flatten()

    for idx, key in enumerate(samples):
        data = samples[key]
        ax = axes[idx]

        # Plot histogram
        ax.hist(data, bins=50, density=True)
        ax.set_title(key)
        ax.set_ylabel('Density')
        ax.set_xlabel(key)

    # Remove any empty subplots
    for j in range(len(samples), len(axes)):
        fig.delaxes(axes[j])
    plt.show()

# Execute the plot function
plot_priors()
prior("alpha")

In [ ]:
simulator_alpha = OutbreakSimulator(variant='alpha')
simulator_omicron = OutbreakSimulator(variant='omicron')

In [ ]:
plot_incubation_distribution(simulator_alpha.shapeIncub, simulator_alpha.scaleIncub,
                             simulator_alpha.shapeIncubAsymp, simulator_alpha.scaleIncubAsymp)
plot_generation_time_distribution(simulator_alpha.shape_generation_time, simulator_alpha.scale_generation_time)
plot_delay_distribution(simulator_alpha.delayDist)

In [ ]:
%%time
test_params = prior("alpha")
print(test_params)
#test = simulator_alpha(**test_params, return_df=True)
#test['sim_data_df'].head()
#test['sim_data_df'].infect_status.value_counts()

## Neural Posterior Estimation

In [ ]:
adapter = (
    bf.adapters.Adapter()
    .drop(['selection_procedure', 'variant'])  # drop strings
    .to_array()
    .one_hot('selection_procedure_id', num_classes=3)  # must be before convert_dtype
    .convert_dtype(from_dtype="float64", to_dtype="float32")

    .constrain('beta', lower=0, inclusive='none', method="softplus")  # standard
    .constrain([k for k in list(param_names.keys()) if k != 'delta' and k != 'beta'], lower=0, inclusive='none', method="exp")
    .concatenate(list(param_names.keys()), into="inference_variables")
    .standardize('inference_variables')

    .rename('selection_procedure_id', to_key="inference_conditions")
    #.concatenate(['selection_procedure_id', 'variant_id'], into="inference_conditions")

    .rename('sim_data', to_key="summary_variables")
)

In [ ]:
from bayesflow.utils.serialization import serializable

@serializable("bayesflow.networks")
class DoubleSummaryNetwork(bf.networks.SummaryNetwork):
    def __init__(self, inner_network, outer_network, name=None, **kwargs):
        super().__init__(**kwargs)
        self.name = 'inner_' + inner_network.name + '_outer_' + outer_network.name if name is None else name
        self.inner_network = inner_network  # operates over elements
        self.outer_network = outer_network  # operates over observations

    def call(self, x, training: bool = False, **kwargs):
        b_size, n_outer_obs, n_inner_obs = keras.ops.shape(x)[:3]

        # Flatten to combine batch and outer observation dimensions
        x_flat = keras.ops.reshape(x, (b_size * n_outer_obs, n_inner_obs, *keras.ops.shape(x)[3:]))

        # Apply the inner network to each element in the outer observation
        inner_output = self.inner_network(x_flat, training=training, **kwargs)

        # Reshape back to (b_size, n_outer_obs, inner_output_dim)
        inner_output = keras.ops.reshape(inner_output, (b_size, n_outer_obs, *keras.ops.shape(inner_output)[1:]))

        # Apply the outer network to the inner outputs
        outer_output = self.outer_network(inner_output, training=training, **kwargs)
        return outer_output


    def get_config(self):
        config = super().get_config()
        config.update({
            "inner_network": self.inner_network,
            "outer_network": self.outer_network
        })
        return config

In [ ]:
job_array_id = 9 # flow matching: 18, coupling flow: 9
nrmse_list = []
calibration_list = []
model_list = []
sum_i = 0  # transformer summary
for net_id, (inf_i, in_summary_dim, out_summary_dim) in enumerate(itertools.product([0, 1], [8, 18, 24, 36], [18, 24, 36])):
    if net_id != job_array_id:
        continue
    if inf_i == 0:
        epochs = 100  # coupling flow
    elif inf_i == 1:
        epochs = 300  # flow matching
    elif inf_i == 2:
        epochs = 300  # consistency model
    else:
        raise ValueError(f"Unknown inference network index: {inf_i}")

    summary_network = [
        DoubleSummaryNetwork(
             inner_network=bf.networks.TimeSeriesNetwork(summary_dim=in_summary_dim, dropout=0.1, recurrent_dim=32),
             outer_network=bf.networks.SetTransformer(summary_dim=out_summary_dim, dropout=0.1),
             name=f'transformer_time_series'
        ),
        DoubleSummaryNetwork(
             inner_network=bf.networks.TimeSeriesNetwork(summary_dim=in_summary_dim, dropout=0.1, recurrent_dim=32),
             outer_network=bf.networks.DeepSet(summary_dim=out_summary_dim, dropout=0.1,
                                               mlp_widths_equivariant=(128, 128)),
             name=f'deep_set_time_series'
        )
    ][sum_i]

    inference_network = [bf.networks.CouplingFlow(depth=7, subnet_kwargs={"dropout": 0.1}, transform='spline'),
                         bf.networks.FlowMatching(subnet_kwargs={"dropout": 0.1}, use_optimal_transport=True,
                                                  integrate_kwargs={'steps': 200}),
                         #bf.networks.ConsistencyModel(epochs*num_training_batches*batch_size),
                         ][inf_i]

    model_name = (f'pedcov_{["transformer", "deep_set"][sum_i]}_{["coupling_flow", "flow_matching", "consistency_model"][inf_i]}'
                  f'_{in_summary_dim}_{out_summary_dim}.keras')

    workflow = bf.BasicWorkflow(
        adapter=adapter,
        summary_network=summary_network,
        inference_network=inference_network
    )

    model_path = f'models/{model_name}'
    print(model_path)
    if os.path.exists(model_path):
        workflow.approximator = keras.saving.load_model(filepath=model_path)
    else:
        raise FileNotFoundError(model_path)
    #diagnostics = workflow.compute_default_diagnostics(test_data=validation_data, num_samples=300)
    #nrmse_list.append(diagnostics.loc['NRMSE'].mean())
    #calibration_list.append(diagnostics.loc['Calibration Error'].mean())
    #model_list.append(model_name)

# Apply trained model to real data

In [ ]:
# # create test data to validate code
# test_params = prior("alpha")
# test_df = simulator_alpha(**test_params, return_df=True)['sim_data_df']
# test_df.to_csv('PedCov/data/test_data.txt', sep=' ', index=False)

In [ ]:
# specify the PedCov path
variants = ['alpha', 'omicron'][:1]  # note: so far only trained for alpha
data_path_alpha = 'PedCov/data/test_data.txt'  # todo: exchange with real PedCov of alpha
data_path_omicron = None # 'PedCov/data/test_data.txt'
num_samples = 1000

In [ ]:
# load the PedCov
dfs = {
    'alpha': pd.read_csv(data_path_alpha, delimiter=' ') if data_path_alpha is not None else None,
    'omicron': pd.read_csv(data_path_omicron, delimiter=' ') if data_path_omicron is not None else None
}

# patient id in household
dfs['alpha']['id_patient'] = dfs['alpha'].groupby("id_hh").cumcount() + 1  # Row number within id_hh
if data_path_omicron is not None:
    dfs['omicron']['id_patient'] = dfs['omicron'].groupby("id_hh").cumcount() + 1

real_data_results = {
    'alpha': None,
    'omicron': None
}

# prepare the PedCov for neural networks
for variant in variants:
    if variant == 'alpha':
        simulator_bf = simulator_alpha
    else:
        simulator_bf = simulator_omicron
    household_data = normalize_household_data(dfs[variant], minimal_length=simulator_bf.minimal_length)[np.newaxis]
    real_data_results[variant] = {
        'sim_data': household_data,
        'selection_procedure': ['pedcov'],
        'selection_procedure_id': [1],  # pedcov
        'variant': [variant],
        'variant_id': [0] if variant == 'alpha' else [1],  # 0 for alpha, 1 for omicron
    }

# get posterior samples
for variant in variants:
    print(f"Variant: {variant}")
    if variant == 'alpha':
        simulator_bf = simulator_alpha
    else:
        simulator_bf = simulator_omicron

    posterior_samples_real = workflow.sample(conditions=real_data_results[variant], num_samples=num_samples)
    real_data_results[variant]['posterior_samples'] = posterior_samples_real

    stan_posterior_samples = get_stan_posterior(dfs[variant], param_names, simulator_bf, show_progress=True,
                                                use_simple_model=True)  # todo: set to False, True is only for testing

    # thin samples to num_samples
    sample_idx = np.random.choice(len(stan_posterior_samples[list(param_names.keys())[0]]), num_samples)
    for p in param_names.keys():
        stan_posterior_samples[p] = stan_posterior_samples[p][sample_idx].reshape(1, num_samples, 1)  # select samples
    real_data_results[variant]['stan_posterior_samples'] = stan_posterior_samples

    # save samples
    with open(f'plots/pedcov_real_posterior_samples_{variant}.pickle', 'wb') as f:
        pickle.dump(real_data_results, f)

In [ ]:
# plot posterior samples vs prior
for variant in variants:
    print(f"Variant: {variant}")
    fig = bf.diagnostics.pairs_posterior(
        estimates=real_data_results[variant]['posterior_samples'],
        variable_keys=list(param_names.keys()),
        variable_names=list(param_names.values()),
        label_fontsize=22,
        legend_fontsize=24
    )
    plt.savefig(f'plots/pedcov_real_posterior_{variant}.png', bbox_inches='tight')
    plt.show()

    fig = bf.diagnostics.pairs_posterior(
        estimates=real_data_results[variant]['stan_posterior_samples'],
        variable_keys=list(param_names.keys()),
        variable_names=list(param_names.values()),
        label_fontsize=22,
        legend_fontsize=24
    )
    plt.savefig(f'plots/pedcov_real_stan_posterior_{variant}.png', bbox_inches='tight')
    plt.show()

In [ ]:
# plot credible intervals for each parameter and each variant
for variant in variants:
    posterior_samples = np.stack([real_data_results[variant]['posterior_samples'][p][0, :, 0]
                                  for p in param_names.keys()], axis=-1)
    ax = sampling_parameter_cis(posterior_samples, alpha=[99, 95, 80],
                                param_names=list(param_names.values()), title=f"Real Data NPE Posterior CIs - {variant}")
    # add vertical line at 1 for the mu parameters
    ax.vlines(1, ymin=1.75, ymax=8.25, color='grey', linestyle='--')
    plt.savefig(f'plots/pedcov_real_CIs_{variant}.png', bbox_inches='tight')
    plt.show()

    stan_posterior_samples = np.stack([real_data_results[variant]['stan_posterior_samples'][p][0, :, 0]
                                       for p in param_names.keys()], axis=-1)
    ax = sampling_parameter_cis(stan_posterior_samples, alpha=[99, 95, 80],
                                param_names=list(param_names.values()), title=f"Real Data STAN Posterior CIs - {variant}")
    # add vertical line at 1 for the mu parameters
    ax.vlines(1, ymin=1.75, ymax=8.25, color='grey', linestyle='--')
    plt.savefig(f'plots/pedcov_real_stan_CIs_{variant}.png', bbox_inches='tight')
    plt.show()